# Part B - Prepare the Data (Pathway A1: PDF/TXT)

Run **Cell 1** and **Cell 3** from your lab sheet first (they define `RAW_DIR`, `BASE_DIR` and `PROCESSED_DIR`). Then upload your PDFs into `RAW_DIR` and run the cells below in order.

# Cell 1 — Initialise the Colab workspace

In [32]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


## Cell 2 — Extract the dummy ZIP

In [33]:
# Cell 2 — Extract practice ZIP
# OPTIONAL: Run only if using the dummy dataset

from zipfile import ZipFile

zip_path = RAW_DIR / "Topic_3_Dummy_Datasets.zip"

if zip_path.is_file():
    with ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)

    print("Dummy dataset pack extracted.")
    print("Location:", RAW_DIR / "topic3_dummy_datasets")
else:
    print("No ZIP found.")
    print("Continue with your separately uploaded project files.")


No ZIP found.
Continue with your separately uploaded project files.


## Cell 3 — Confirm uploaded inputs

In [34]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
 	print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")


Input count: 1
week3_data_inventory.pdf bytes: 122863
Upload verified. Select your relevant input(s) for Step A3.


## Cell 4 — REQUIRED: Source Selection + Lineage Utilities

In [35]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")


Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


## Cell 4A — Initialise Common Lineage Registry

In [36]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


Common lineage registry ready.
Registered entries: 0


## Cell 5A1 — CONDITIONAL REQUIRED: PDF/TXT ONLY

In [37]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Example: Synthetic practice PDF
# Replace with your actual project file where applicable.
f = select_source(relative_path="week3_data_inventory.pdf")



# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")


Source: week3_data_inventory.pdf
Pages/segments: 6
Extracted characters: 6877
Cleaned characters: 6877
Content sample: 'FloodWise MY\n1. Required Data\nRequirement /\nObjective\nNeeded information /\nfeature\nCandidate\nsource\nWhy usable / constraint\nO1: Document-\ngrounded ﬂood\npreparedness\nassistant\nMalaysian ﬂood reports,\npreparedness\ninformation, evacuation\ninformation and disaster\nupdates\nReliefWeb:\nMalaysia Floods\nNov '

--- Chunking Comparison ---
Fixed chunks: 14
Paragraph-aware chunks: 14

Fixed sample:
'FloodWise MY\n1. Required Data\nRequirement /\nObjective\nNeeded information /\nfeature\nCandidate\nsource\nWhy usable / constraint\nO1: Document-\ngrounded ﬂood\npreparedness\nassistant\nMalay'

Paragraph-aware sample:
'FloodWise MY\n1. Required Data\nRequirement /\nObjective\nNeeded information /\nfeature\nCandidate\nsource\nWhy usable / constraint\nO1: Document-\ngrounded ﬂood\npreparedness\nassistant\nMalay'

PATHWAY A1 completed successfully.


# Cell 5A2 — CONDITIONAL REQUIRED: AFTER A1

In [38]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 14
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 10674 bytes
Validation: PASS
First record: {'record_id': 'week3_data_inventory-0001', 'text': 'FloodWise MY\n1. Required Data\nRequirement /\nObjective\nNeeded information /\nfeature\nCandidate\nsource\nWhy usable / constraint\nO1: Document-\ngrounded ﬂood\npreparedness\nassistant\nMalaysian ﬂood reports,\npreparedness\ninformation, evacuation\ninformation and disaster\nupdates\nReliefWeb:\nMalaysia Floods\nNov 2025 and\nMalaysia Floods\nNov 2024\nProvides Malaysia-speciﬁc\ndisaster reports and ﬂood\nsituation information.\nReports are historical and\nmust not be treated as\nlive ﬂood conditions.\nO1, O2: Retrieve\nre', 'source': 'week3_data_inventory.pdf', 'source_sha256': '9603c1451bfc875fc012ff58b8ccddb8c1f593bc25522e57f9ef2652368a9466', 'processing': 'paragraph-aware-char-500', 'page': None}


# B3 — Validate the prepared result and compare one decision

In [39]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 10674
Validation: file exists and is non-empty; verify content manually too.


# B4.1 — Register Pathway A

In [40]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: week3_data_inventory.pdf -> prepared_text.jsonl


# B4.2 — Generate Common Lineage Manifest

In [41]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


# B4.3 — Validate the Saved Manifest

In [42]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


# C1 — Implement and Run the Integrated V1 Data Pipeline

In [43]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


# Cell 8A — CONDITIONAL REQUIRED: PDF / TXT

In [44]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    "week3_data_inventory.pdf"
)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]

    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


# Cell 9 — REQUIRED: Repeatability and provenance checks

In [45]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
